# Data Exploration
Having a look at the raw data, preprocessing before ML pipeline. 

In [ ]:
# Importing libraries
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sys

In [ ]:
# To be able to access src
sys.path.append(str(Path().resolve().parent))

In [ ]:
import importlib
import src.preprocessing, src.models, src.utils, src.train, src.evaluate, src.visualisation
importlib.reload(src.preprocessing)
importlib.reload(src.models)
importlib.reload(src.utils)
importlib.reload(src.train)
importlib.reload(src.evaluate)
importlib.reload(src.visualisation)

In [ ]:
# Looking at sample data
BASE_DIR = Path("..") # go up one level
DATA_DIR = BASE_DIR / "pur-1-data-main" # base data folder

# Looking at one example 
csv_path = DATA_DIR / "data" / "Task 1" / "Public Dataset" / "Gang_Lowers" / "Shutdown_0_36774.csv"

df = pd.read_csv(csv_path)
df.head()

## Following Gerion's Hands-on ML approach

From Hands-on ML (Gerion, 2019)

Explore the Data
Note: try to get insights from a field expert for these steps.
1. Create a copy of the data for exploration (sampling it down to a
manageable size if necessary).
2. Create a Jupyter notebook to keep a record of your data
exploration.
3. Study each attribute and its characteristics:

- Name
- Type (categorical, int/float, bounded/unbounded, text,
structured, etc.)
- % of missing values
- Noisiness and type of noise (stochastic, outliers,
rounding errors, etc.)
- Usefulness for the task
- Type of distribution (Gaussian, uniform, logarithmic,
etc.)

4. For supervised learning tasks, identify the target attribute(s).
5. Visualize the data.
6. Study the correlations between attributes.
7. Study how you would solve the problem manually.

8. Identify the promising transformations you may want to apply.
9. Identify extra data that would be useful (go back to “Get the
Data”).
10. Document what you have learned.

In [ ]:
# Load data, combine to one big dataframe
from src.preprocessing import load_csvs, combine_csvs

folder_1 = Path("..") / "pur-1-data-main" / "data" / "Task 1" / "Public Dataset" / "Gang_Lowers"
folder_2 = Path("..") / "pur-1-data-main" / "data" / "Task 1" / "Public Dataset" / "SCRAMS"

dfs = load_csvs(folder_1, folder_2)
# shape 43x800x13

combined_df = combine_csvs(dfs)
# shape 34400x13

### Study each feature and summarise

In [ ]:
# Study each feature and its characteristics
combined_df.head()

In [ ]:
# Summarise
combined_df.columns # column names 
feature_cols = ['Channel 4 flux', 'DAS health index', 'Downstream conductivity',
       'Downstream water temperature', 'FC voltage supply', 'SS1 state',
       'SS1 position', 'SS2 state', 'SS2 position',
       'Upstream water temperature']
combined_df[feature_cols].describe()

# DAS health index always == 1 ? Can be removed if so.
# FC voltage supply always == 1 ?

### Feature distribution

In [ ]:
# Histograms of feature distributions
from src.visualisation import histogram_plotting

histogram_plotting(combined_df, feature_cols, bins=10) 

### Visualisation

In [ ]:
# Example events 
from src.visualisation import plot
plot(combined_df, feature_cols=feature_cols, ID=1031264) # Gang Lower shutdown
plot(combined_df, feature_cols=feature_cols, ID=101938) # SCRAM shutdown

# Further demonstrating the need to remove DAS health index and FC voltage supply, as they are constant and do not provide any information.

In [ ]:
# Want to compare the feature time series of the different shut down IDs to see if some IDs have outlier-like behaviour

# Amateurish code:

gang_lower_subset = combined_df[combined_df['target']==0]
SCRAM_subset = combined_df[combined_df['target']==1]

# gang lower
for col in feature_cols: 
    plt.title(f"Gang Lower - {col}")
    for ID in gang_lower_subset['ID'].unique():
        subset = gang_lower_subset[gang_lower_subset['ID']==ID]
        plt.plot(subset['timestamp'], subset[col], label=ID)
    plt.legend()
    plt.show()

# SCRAM
for col in feature_cols: 
    plt.title(f"SCRAM - {col}")
    for ID in SCRAM_subset['ID'].unique():
        subset = SCRAM_subset[SCRAM_subset['ID']==ID]
        plt.plot(subset['timestamp'], subset[col], label=ID)
    plt.legend()
    plt.show()
    

There are definitely outlier-like behaviour. As of yet, I have not removed any events before model training. There is not a lot of data, and I don't want to disregard any events yet. 
- But could I perhaps identify the IDs that are outlier-like? How?

### Missing Values

In [ ]:
# missing values
combined_df.isna().sum() # Have 5 missing values for all given features

In [ ]:
# Where are we missing values? If short stretch --> fill forward could perhaps be used?

# https://stackoverflow.com/questions/30447083/python-pandas-return-only-those-rows-which-have-missing-values
null_data = combined_df[combined_df.isnull().any(axis=1)]
null_data 
# Looks like there are two affected events, ID 23769 (timestamp 101-112) and ID 2784 (timestamp 511-519)

In [ ]:
# Missing data IDs
ID_1 = 23769
ID_2 = 2784

In [ ]:
# fill with -1 so the missing data is shown clearly
filled_null_data = combined_df.fillna(value=-1)

time_delta = 20 # want to see 20 seconds before and after the events 
plot(filled_null_data, feature_cols=feature_cols, ID=ID_1, time_start=101-time_delta, time_end=112+time_delta) 
plot(filled_null_data, feature_cols=feature_cols, ID=ID_2, time_start=511-time_delta, time_end=519+time_delta) 

# Looks like a forwardfill would work well.
# But perhaps an average of the previous and the next value is better? 


In [ ]:
# Forward filling initially now, can change it later if I want
combined_df.ffill(inplace=True)

In [ ]:
plot(combined_df, feature_cols=feature_cols, ID=ID_1, time_start=101-time_delta, time_end=112+time_delta) 
plot(combined_df, feature_cols=feature_cols, ID=ID_2, time_start=511-time_delta, time_end=519+time_delta) 
# plotting the two events after forward fill
# We see that in fact there was more variation in the data than what I first thought (fooled by relative difference on the y-scale)
# Still decide to keep ffill as the missing values are only for a short stretch

### Feature correlation

In [ ]:
from src.visualisation import plot_correlation_matrix, plot_feature_correlation_with_target

plot_correlation_matrix(combined_df[feature_cols], combined_df['target'], max_features=8)
plot_feature_correlation_with_target(combined_df[feature_cols], combined_df['target'], max_features=8)

In [ ]:
# Indicative of SCRAM:
# High Channel 4 flux
# Low downstream conductivity 
# Low downstream water temperature
# Low upstream water temperature
 
# SS1 and SS2 state and position have less impact on target (values < abs(0.10), meaning less correlation)

# Note: DAS health index and FC voltage supply have no correlation values, because they are constants.

# Also note how some of the descriptive features are correlated, such as downstream water temperature and downstream conductivity
# Indicative of redundant features. 

A note: This feature correlation analysis does not take the time-dependent nature of the features into account. Looks at each timestep, and not each event! Need to do feature extraction/flattening and re-attempt? 

In [ ]:
# Should present feature extraction function in more depth?

In [ ]:
# Feature extraction, then feature correlation
from src.utils import extract_features
X, y = extract_features(dfs, feature_cols)

In [ ]:
from src.visualisation import plot_correlation_matrix, plot_feature_correlation_with_target
plot_correlation_matrix(X, y, max_features=20)
plot_feature_correlation_with_target(X, y, max_features=20)

Analysis of correlation matrix and correlation with target of feature extracted data tells us that 
- larger Channel 4 flux std 
- larger Channel 4 flux max
- smaller Channel 4 flux slope (more negative)
- larger Channel 4 flux mean
As well as
- smaller SS2 position mean 
- larger SS1 position max

...

Are indicative of SCRAM shutdowns over Gang Lowers (SCRAM=1, GL=0)

In [ ]:
from src.utils import flatten_features
X_flat, y = flatten_features(dfs, feature_cols)
plot_correlation_matrix(X_flat, y, max_features=30)
plot_feature_correlation_with_target(X_flat, y, max_features=20)

Analysis of correlation matrix and correlation with target of feature flattened data tells us that 
- SS1 and SS2 state at timestep 583-585
- Channel 4 flux at around timestep 300

Are most important for determining shut down event.

In [ ]:
# Only taking every 80th timestamp, to reduce the number of features and make it more manageable for the correlation matrix.

In [ ]:
from src.utils import flatten_features_resampled_timesteps
X_flat_reduced, y = flatten_features_resampled_timesteps(dfs, feature_cols, step=80)
plot_correlation_matrix(X_flat_reduced, y, max_features=30)
plot_feature_correlation_with_target(X_flat_reduced, y, max_features=30)

# Going back for some more exploration
### Want some more informative plots for the report

#### Raw plots

In [ ]:
feature_cols_for_plotting = ['Channel 4 flux', 'Downstream conductivity', 
                             'Downstream water temperature', 'SS1 state', 
                             'SS1 position', 'SS2 state', 'SS2 position', 
                             'Upstream water temperature']

In [ ]:
# Raw plots
from src.visualisation import plot_raw_timeseries
plot_raw_timeseries(combined_df, feature_cols_for_plotting, target_col="target", id_col="ID", time_col="timestamp", class_value=0, title="Gang Lower - Raw Time Series", ncols=2)
plot_raw_timeseries(combined_df, feature_cols_for_plotting, target_col="target", id_col="ID", time_col="timestamp", class_value=1, title="SCRAM - Raw Time Series", ncols=2)

#### Outliers

In [ ]:
# Create subsets
gang_lower_subset = combined_df[combined_df['target']==0]
SCRAM_subset = combined_df[combined_df['target']==1]

In [ ]:
# Outliers 
from src.utils import top_outliers

outlier_df_gang_lower = top_outliers(combined_df[combined_df['target']==0], feature_cols)
outlier_df_SCRAM = top_outliers(combined_df[combined_df['target']==1], feature_cols)

# plot
# from src.visualisation import top_outlier_plots

# print("Gang lower:")
# top_outlier_plots(gang_lower_subset, outlier_df_gang_lower, feature_cols_for_plotting, top_n=2)
# print("SCRAM:")
# top_outlier_plots(SCRAM_subset, outlier_df_SCRAM, feature_cols_for_plotting, top_n=0, chosen_ID=26354)


In [ ]:
# Alternative visualisation
from src.visualisation import top_outlier_plots_grid
print("Gang lower:")
top_outlier_plots_grid(gang_lower_subset, outlier_df_gang_lower, feature_cols_for_plotting, top_n=2)
print("SCRAM:") 
top_outlier_plots_grid(SCRAM_subset, outlier_df_SCRAM, feature_cols_for_plotting, top_n=0, chosen_ID=26354) 

#### Mean and std plots

In [ ]:
# Mean and std plots
from src.visualisation import plot_mean_std_timeseries
plot_mean_std_timeseries(combined_df, feature_cols_for_plotting, target_col="target", time_col="timestamp", label_0="Gang Lower", label_1="SCRAM")

### Misclassified events (after model training)


In [ ]:
# plot misclassified events
event_ids_gang_lower = [1118982, 674296, 193108] # From test.ipynb
event_ids_SCRAM = [10413] # From test.ipynb

from src.visualisation import top_outlier_plots_grid
print("Gang lower:")
top_outlier_plots_grid(gang_lower_subset, outlier_df_gang_lower, feature_cols_for_plotting, top_n=0, chosen_ID=event_ids_gang_lower)

print("SCRAM:")
top_outlier_plots_grid(SCRAM_subset, outlier_df_SCRAM, feature_cols_for_plotting, top_n=0, chosen_ID=event_ids_SCRAM)


In [ ]:
# Have a look at the flatlines
ID_flatline = 859370
plot(combined_df, feature_cols=feature_cols, ID=ID_flatline)

In [ ]:
# SCRAM outlier
ID_flatline = 26354
plot(combined_df, feature_cols=feature_cols, ID=ID_flatline)

In [ ]:
A = combined_df[combined_df['ID']==26354]
A = A[480 < A['timestamp']]
A = A[A['timestamp'] < 500]

plt.plot(A['timestamp'], A['Downstream water temperature'])
plt.plot(A['timestamp'], A['SS1 position'])
plt.plot(A['timestamp'], A['SS2 position'])

A